# Семинар 16. Python. Работа с аудио и видео

## Мотивация

Модель не умеет слушать wav. Чтобы обучить распознавание речи, детектор поломки
станка по звуку или классификатор жанра, запись сначала превращают в **признаки**
— почти всегда в спектрограмму, то есть в картинку «частота × время». Весь путь от
файла до этой картинки лежит на вас, и на нём три места, где данные молча портятся.

1. **Частота дискретизации.** `librosa.load` по умолчанию приводит любой файл к
   22050 Гц и к моно. Половину записей это устраивает, а у второй половины
   бесследно исчезает всё выше 11 кГц — и модель учится не на том, что вы думали.
2. **Форма массива.** У `soundfile` стерео — это `(сэмплы, каналы)`, у `librosa` и
   `torchaudio` — `(каналы, сэмплы)`. Перепутали — и «длительность записи» стала
   равна 0.00009 секунды. Исключения при этом не будет.
3. **Одного сигнала во времени мало.** По графику амплитуды не видно, какие
   частоты звучали. Нужен переход в частотную область — и сразу выясняется, что
   спектр всей записи целиком тоже не годится, если звук меняется.

Сегодня проходим путь целиком: сэмплы и частота дискретизации → чтение и запись
wav → амплитуда и клиппинг → спектр → STFT и спектрограмма → шкала дБ и мел-шкала
→ те же операции тензорами в `torchaudio` → коротко видео как последовательность
кадров. Домашка — «спектрограмма аудиозаписи», так что получившийся конвейер
пригодится сразу.

Что уже знакомо и повторяться не будет: массивы `numpy` и графики `matplotlib`
(семинар 1, практикум по `numpy` — ещё и семинар 14), работа с изображениями
(семинар 15). Спектрограмма и
кадр видео — ровно такие же двумерные и трёхмерные массивы, как картинка, и всё,
что вы умеете делать с изображением, применимо и к ним.

## 1. Звук — это массив чисел

Микрофон измеряет давление воздуха и записывает результат числами через равные
промежутки. Одно измерение — **сэмпл**, число измерений в секунду — **частота
дискретизации** (sample rate, `sr`, Гц). Отсюда главное соотношение семинара:

`длительность = число сэмплов / sr`

Сгенерируем чистый тон — синус частоты 440 Гц (нота «ля»).

In [ ]:
import numpy as np

sr = 22050                                # частота дискретизации: сколько чисел в секунде
t = np.arange(sr * 2) / sr                # моменты времени двух секунд, шаг 1/sr
y = 0.5 * np.sin(2 * np.pi * 440 * t)     # тон 440 Гц, амплитуда 0.5 — тише максимума
print(y.shape, y.dtype)                   # обычный одномерный массив float64
print(len(y) / sr, "секунд")              # длительность = число сэмплов / частота

`t` — моменты времени сэмплов с шагом `1/sr`, массив `y` — сам звук. Никакого
«аудиоформата» в памяти нет: это обычный массив чисел от -1 до 1.
Тип здесь `float64`, потому что так посчитал `numpy`, но в общем случае он
зависит от того, кто отдал массив: `soundfile` вернёт `float64`, а `librosa` и
`torchaudio` — `float32` (увидим в разделах 4 и 10). Диапазон -1..1 при этом
одинаковый у всех.

Посмотрим на первые 5 миллисекунд.

Из волны в массив чисел

In [ ]:
import matplotlib.pyplot as plt

plt.plot(t[:110], y[:110])    # 110 сэмплов ≈ 5 мс: на всей секунде синус слился бы в заливку
plt.xlabel("время, с")
plt.ylabel("амплитуда")
plt.show()

И, раз это семинар про звук, тон можно послушать прямо в ноутбуке —
`IPython.display.Audio` делает плеер из массива и частоты дискретизации.

In [ ]:
from IPython.display import Audio

Audio(y, rate=sr)             # плеер прямо в ноутбуке: массив + частота дискретизации

#### ❓ **Вопрос**: Тот же звук сгенерировали с `sr = 44100`. Как изменятся форма массива и длительность?

<details>

<summary><strong>Ответ</strong></summary>

Форма станет `(88200,)` — чисел вдвое больше, потому что вдвое больше измерений в секунду. Длительность не изменится: она считается как `len(y) / sr`, и в дроби выросли и числитель, и знаменатель — `88200 / 44100 = 2.0`, как в ячейке выше было `44100 / 22050 = 2.0`.

На графике при этом на тех же 5 мс окажется вдвое больше точек: форма волны описана подробнее.

</details>

## 2. Запись и чтение wav: soundfile

Массив в файл и обратно проще всего гонять библиотекой `soundfile` (обёртка над
libsndfile). Формат `wav` — это небольшой заголовок и подряд идущие сэмплы; самый
ходовой вариант хранения — 16-битные целые, `PCM_16`.

Почему `soundfile`, а не `wave` из стандартной библиотеки и не
`scipy.io.wavfile`: `wave` отдаёт сырые байты, разбирать их в числа надо самому;
`scipy` возвращает то, что лежит в файле, — для `PCM_16` это целые `int16`;
`soundfile` читает не только wav и сразу приводит сэмплы к `float64` в диапазоне
-1..1, то есть к тому виду, в котором их и обрабатывают.

<details><summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>Откуда взялось некруглое 44100. В конце семидесятых цифровой звук негде былохранить: единственным доступным носителем с нужной скоростью потока былавидеокассета, и звук писали в неё, притворяясь видеосигналом. В строкупомещалось три сэмпла, отсюда и число: 245 строк × 60 полей в секунду (NTSC)= 44100, и ровно столько же даёт 294 × 50 для PAL — редкое совпадение, из-закоторого стандарт получился общим. Так частота компакт-диска оказаласьнаследством видеомагнитофона.А 22050, с которым мы работаем на семинаре, — это ровно половина: так корочемассивы и быстрее считается, а всё, что ниже 11 кГц, сохраняется.</details>

In [ ]:
import soundfile as sf
from pathlib import Path

work = Path.home() / "seminar-16"            # работаем в домашнем каталоге, не в /tmp
work.mkdir(exist_ok=True)
sf.write(work / "tone.wav", y, sr, subtype="PCM_16")   # PCM_16 — два байта на сэмпл
print((work / "tone.wav").stat().st_size, "байт")      # 44100·2 + 44 байта заголовка

44100 сэмплов по 2 байта плюс 44 байта заголовка — размер файла сходится
руками. Прочитаем обратно.

In [ ]:
data, sr_read = sf.read(work / "tone.wav")   # вернётся пара: массив и его частота
print(data.shape, data.dtype, sr_read)
print(float(np.abs(data - y).max()))        # не ноль: сэмплы округлились при записи

`sf.read` возвращает **пару** «массив, частота дискретизации»: частота живёт в
файле, а не в массиве, и потерять её нельзя. Сэмплы по умолчанию приводятся к
`float64`, а целочисленные форматы (`PCM_16`, `PCM_24`) при этом делятся на свой
максимум и попадают в -1..1. Файлы с `subtype='FLOAT'` читаются как есть: если
туда записали 2.5, столько и вернётся.

#### ❓ **Вопрос**: Прочитанный массив не совпал с исходным в точности. Откуда взялось расхождение порядка 3e-5?

<details>

<summary><strong>Ответ</strong></summary>

Это квантование: сэмплы легли на сетку из 65536 уровней с шагом `1/32768 ≈ 3.05e-5`, и напечатанное расхождение — как раз порядка шага. Заметьте, что получилось почти ровно `3.05e-5`, а не половина: округляй libsndfile к ближайшему уровню, ошибка не превышала бы полшага, но он округляет **вниз**, и промах доходит до целого шага. У положительных сэмплов значение от этого уменьшается, у отрицательных — растёт по модулю.

Если потери недопустимы (например, вы сохраняете промежуточный результат обработки), пишите `subtype='FLOAT'` — тогда расхождение будет нулевым, но файл станет вдвое больше.

</details>

Само превращение `float64` в 16-битные целые называется **квантованием**: libsndfile умножает сэмпл на 32768 и **округляет вниз**, к ближайшему меньшему уровню сетки (не к ближайшему вообще). Поэтому значение всегда смещается вниз, и промах доходит почти до целого шага `1/32768`. Обратно исходные числа уже не достать.

Метаданные можно узнать, не читая сами сэмплы, — это важно, когда файл на
несколько часов и в память целиком не влезает.

In [ ]:
info = sf.info(work / "tone.wav")           # только заголовок, файл не читается целиком
print(info.samplerate, info.channels, info.frames)
print(info.duration, info.subtype)          # duration в секундах, subtype — формат сэмплов

А если из такого файла нужны не метаданные, а кусок, читают именно кусок: у
`sf.read` для этого есть `start` и `stop` (номера сэмплов), у `librosa.load` —
`offset` и `duration` в секундах. В память попадёт только запрошенное.


In [ ]:
# start и stop — номера сэмплов: берём вторую секунду, а не весь файл
piece, sr_piece = sf.read(work / "tone.wav", start=sr, stop=sr + sr // 2)
print(piece.shape, sr_piece, len(piece) / sr_piece)   # полсекунды

## 3. Моно и стерео: форма массива

**Канал** — отдельная дорожка звука (левая, правая). Соберём стереофайл: слева
440 Гц, справа 880 Гц.

Две раскладки стереомассива

In [ ]:
right = 0.25 * np.sin(2 * np.pi * 880 * t)      # правый канал: октавой выше и тише
stereo = np.stack([y, right], axis=1)          # axis=1 — канал становится вторым измерением
sf.write(work / "stereo.wav", stereo, sr, subtype="PCM_16")
print(stereo.shape)                            # (сэмплы, каналы)

Файл записан. Читаем его обратно и смотрим, в какой форме массив вернётся
и что про файл говорят метаданные.

In [ ]:
st, _ = sf.read(work / "stereo.wav")
print(st.shape)                                # у soundfile канал — последнее измерение
meta = sf.info(work / "stereo.wav")
print(meta.channels, meta.frames)              # frames — сэмплов на канал, а не всего

У `soundfile` форма стерео — `(сэмплы, каналы)`: канал идёт **последним**
измерением, а `frames` в метаданных — это число сэмплов на канал, а не общее число
чисел. Сведение в моно — усреднение по каналам.

In [ ]:
mono = st.mean(axis=1)        # усреднение по каналам — самый простой способ схлопнуть в моно
print(mono.shape)
print(round(float(np.abs(st[:, 0]).max()), 3),    # левый: 0.5, как мы и задали
      round(float(np.abs(st[:, 1]).max()), 3))    # правый: 0.25

#### ❓ **Вопрос**: Скрипт считает длительность как `len(x) / sr`. Что он вернёт для нашего стереофайла и почему на это нельзя полагаться?

<details>

<summary><strong>Ответ</strong></summary>

Здесь повезёт: `len` двумерного массива — длина **первого** измерения, а у `soundfile` первым идёт число сэмплов, значит `44100 / 22050 = 2.0` — верно.

Но стоит прочитать тот же файл библиотекой, кладущей канал первым (это `librosa` с `mono=False` и `torchaudio` — увидим в разделах 4 и 10), как `len` станет равен 2, а длительность — `2 / 22050 ≈ 0.00009` секунды. Исключения не будет, будет тихо неверный результат. Смотреть надо на `.shape` целиком.

</details>

## 4. librosa.load: удобно и опасно

`librosa` — основная библиотека анализа звука в Python. Её `load` читает почти
любой формат, не только wav, но по дороге **меняет данные**. Запишем тон на
«музыкальной» частоте 44100 Гц и прочитаем без аргументов.

In [ ]:
import librosa

t44 = np.arange(44100 * 2) / 44100             # те же две секунды, но частота вдвое выше
y44 = 0.5 * np.sin(2 * np.pi * 440 * t44)      # тон тот же, 440 Гц
sf.write(work / "tone44k.wav", y44, 44100, subtype="PCM_16")
print(len(y44), "сэмплов при 44100 Гц")        # 88200 — вот что лежит в файле

В файле 88200 сэмплов при 44100 Гц. Теперь откроем его самой популярной
функцией — `librosa.load` — и посмотрим, что она вернёт.

In [ ]:
loaded, sr_loaded = librosa.load(work / "tone44k.wav")
print(sr_loaded, loaded.shape, loaded.dtype)   # 22050 и вдвое меньше сэмплов, чем в файле

В файле 88200 сэмплов при 44100 Гц, а получили 44100 при 22050 Гц. Это
умолчание `sr=22050`: `librosa.load` **ресемплирует** запись. Отключается явным
`sr=None` — «оставить как в файле».

Ресемплинг вниз — не бесплатная операция: при частоте `sr` представимы только
частоты до `sr/2` (**предел Найквиста**), поэтому при 22050 Гц всё выше 11 кГц
отфильтровывается и исчезает навсегда. Длительность при этом не меняется —
сокращаются и число сэмплов, и частота.

In [ ]:
native, sr_native = librosa.load(work / "tone44k.wav", sr=None)   # sr=None — как в файле
print(sr_native, native.shape)
print(librosa.get_duration(y=loaded, sr=sr_loaded),      # длительность одинаковая:
      librosa.get_duration(y=native, sr=sr_native))      # сэмплов меньше, но и частота ниже

#### ❓ **Вопрос**: Массив после `librosa.load` вдвое короче, а `get_duration` в обоих случаях даёт 2.0. Что тогда всё-таки потерялось?

<details>

<summary><strong>Ответ</strong></summary>

Длительность и не могла потеряться: сократились и число сэмплов, и частота дискретизации, дробь `len(y)/sr` осталась прежней. Потерялись **высокие частоты**. При `sr = 22050` представимы частоты только до `sr/2 = 11025` Гц (предел Найквиста), всё выше при ресемплинге отфильтровывается.

Нашему тону 440 Гц это безразлично, а записи с шипящими согласными, тарелками или ультразвуковым датчиком — нет. Поэтому в исследовательском коде пишут `sr=None` или явное нужное значение, а не полагаются на умолчание.

</details>

Второе умолчание — `mono=True`: любые каналы схлопываются в один — тем же
усреднением по каналам, что мы делали руками в разделе 3, а не выбором левой
дорожки. Проверим на нашем стереофайле.

In [ ]:
one, _ = librosa.load(work / "stereo.wav", sr=None)                 # mono=True по умолчанию
both, _ = librosa.load(work / "stereo.wav", sr=None, mono=False)    # каналы сохранены
print(one.shape, both.shape)   # (сэмплы,) против (каналы, сэмплы) — канал стал ПЕРВЫМ

#### ❓ **Вопрос**: `soundfile` вернул для того же файла `(44100, 2)`, а `librosa` с `mono=False` — `(2, 44100)`. Кто из них ошибается?

<details>

<summary><strong>Ответ</strong></summary>

Никто: это разные соглашения. Оба соглашения одинаково законны, wav их не навязывает: внутри файла сэмплы каналов просто чередуются подряд, и как разложить их по осям — решает библиотека. `soundfile` кладёт канал последним измерением, ближе к порядку байт в файле. `librosa` (и, как увидим, `torchaudio`) кладёт канал первым, потому что дальше по конвейеру удобно обрабатывать каждый канал как отдельную строку матрицы.

Практический вывод один: после любой загрузки печатайте `.shape` и сверяйтесь с ожидаемой длительностью — как в вопросе про `len(x) / sr` выше.

</details>

## 5. Амплитуда, нормализация, клиппинг

Диапазон -1..1 — не украшение: целочисленные форматы (`PCM_16` и родня) хранят
именно его, и всё, что вышло за границу, при записи молча срежется по ней —
исключения не будет, будет испорченный звук. (Исключение — `subtype='FLOAT'` из
раздела 2: он пишет что дали, но такие файлы дальше по конвейеру ждут не везде.)
Записи приходят разной громкости, и перед подачей в модель их приводят к одному
масштабу. Самый простой способ — **пик-нормализация**: поделить на максимум
модуля.

Есть и второй способ — **по RMS** (`sqrt(mean(y**2))`, среднеквадратичная
амплитуда). Пик смотрит на один-единственный сэмпл, поэтому случайный щелчок в
записи делает всю её тихой; RMS усредняет по всей записи и ближе к тому, что ухо
воспринимает как громкость. Зато RMS-нормировка не гарантирует, что сигнал
остался в -1..1, — после неё возможен клиппинг, и пик приходится проверять
отдельно. Правило: нужен гарантированный диапазон — пик, нужна сопоставимая
громкость записей в датасете — RMS. RMS считают в задачах B4 и H2.

In [ ]:
quiet = 0.05 * np.sin(2 * np.pi * 220 * t)     # заведомо тихая запись
print(round(float(np.abs(quiet).max()), 4))    # пик 0.05 — до потолка далеко
norm = quiet / np.abs(quiet).max()             # нормализация по пику: делим на максимум
print(round(float(np.abs(norm).max()), 4))     # ровно 1.0 — громче уже нельзя

Нормировка по RMS — это то же деление, только на RMS, а не на пик, и с явной
целевой громкостью: `y / rms * target`. Пик после неё нужно проверять.

In [ ]:
rms = np.sqrt(np.mean(quiet ** 2))       # RMS — «средняя» громкость, а не одиночный пик
by_rms = quiet / rms * 0.1               # приводим к целевому RMS 0.1
print(round(float(np.sqrt(np.mean(by_rms ** 2))), 4))   # цель достигнута
print(round(float(np.abs(by_rms).max()), 4))            # а пик проверяем отдельно

Соблазн «сделать погромче» умножением на константу заканчивается
**клиппингом**: всё, что вышло за диапазон, срезается по границе.

In [ ]:
loud = np.clip(quiet * 30, -1.0, 1.0)    # «сделать погромче» умножением — и срезать хвосты
print(int(np.sum(np.abs(quiet * 30) > 1.0)), "сэмплов из", len(quiet))   # сколько срезано
print(round(float(np.abs(quiet * 30).max()), 3),    # куда сигнал хотел уйти
      round(float(np.abs(loud).max()), 3))          # и где его остановил потолок

Разницу лучше слышно, чем видно: сначала аккуратно нормализованный сигнал,
потом, после паузы, тот же тон с клиппингом.

In [ ]:
pause = np.zeros(sr // 2)     # полсекунды тишины, чтобы отделить один вариант от другого
Audio(np.concatenate([norm, pause, loud]), rate=sr)   # сначала чистый, потом с клиппингом

Посмотрим, во что превратился спектр. Подробно это раздел 6, а пока хватит
одного факта: `np.fft.rfft` раскладывает сигнал на синусы и возвращает по
комплексному числу на частоту, `np.abs` от него — амплитуда этой частоты, а
`np.fft.rfftfreq` — сами частоты в герцах. Печатаем четыре самые громкие.

In [ ]:
spec_loud = np.abs(np.fft.rfft(loud))
grid = np.fft.rfftfreq(len(loud), 1 / sr)
# argsort даёт номера бинов по возрастанию амплитуды, [-4:] — четыре громких
print(sorted(float(grid[i]) for i in np.argsort(spec_loud)[-4:]))

#### ❓ **Вопрос**: Мы срезали верхушки у 23600 сэмплов из 44100. Синус же остался синусом — почему это порча данных?

<details>

<summary><strong>Ответ</strong></summary>

Он как раз перестал быть синусом: у волны появились плоские площадки на уровне ±1, то есть это уже другой сигнал. Последняя ячейка это и показывает: кроме исходных 220 Гц, в спектре появились 660, 1100 и 1540 Гц — нечётные гармоники, которых в `quiet` не было. На слух это хрип.

Поэтому громкость правят делением на пик (или нормировкой по RMS), а не умножением наугад: `norm` из предыдущей ячейки громче ровно в 20 раз и при этом не искажён.

</details>

## 6. Одного времени мало: спектр

Сложим два тона — 440 и 660 Гц. Это **аккорд**: на графике во времени видна волна
сложной формы, и по ней не сказать, из чего она собрана.

In [ ]:
chord = 0.6 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 660 * t)  # два тона сразу
plt.plot(t[:220], chord[:220])   # на глаз это просто волна сложной формы
plt.xlabel("время, с")
plt.ylabel("амплитуда")
plt.show()

Преобразование Фурье раскладывает сигнал по синусам разных частот. Для
вещественного сигнала берут `np.fft.rfft` (вторая половина спектра симметрична и не
нужна), а сетку частот к нему даёт `np.fft.rfftfreq`.

Возвращает `rfft` **комплексные** числа — по одному на частоту. В таком числе
закодированы сразу две вещи: насколько громко звучала частота (модуль, его берёт
`np.abs`) и с каким сдвигом (фаза, к ней вернёмся в разделе 8). Дальше нас будет
интересовать только модуль — амплитуда.

In [ ]:
spectrum = np.abs(np.fft.rfft(chord))       # модуль спектра: насколько сильна каждая частота
freqs = np.fft.rfftfreq(len(chord), 1 / sr)  # какой частоте соответствует каждый бин
top2 = np.argsort(spectrum)[-2:]             # два самых сильных бина
print(sorted(float(freqs[i]) for i in top2))            # ровно 440 и 660 — наши тоны
print(len(spectrum), "бинов, шаг", round(float(freqs[1]), 3), "Гц")

#### ❓ **Вопрос**: Почему пики оказались ровно на 440.0 и 660.0, без дробной части?

<details>

<summary><strong>Ответ</strong></summary>

Шаг частотной сетки равен `sr / N`, то есть `1 / длительность` — из второй строки вывода видно, что он `0.5` Гц (запись длится 2 секунды). И 440, и 660 делятся на 0.5 нацело, поэтому каждая частота попала точно в свой бин.

Если бы тон был 440.3 Гц, энергия размазалась бы между соседними бинами и максимум встал бы на 440.5 — точнее шага сетки ответить нельзя. Улучшить разрешение по частоте можно только взяв более длинный кусок сигнала.

</details>

## 7. Зачем нужен STFT

Аккорд звучал неизменным все две секунды, поэтому спектр всей записи его описал.
Возьмём сигнал, который **меняется**: чирп — тон, чья частота линейно растёт с 200
до 2000 Гц за 4 секунды.

Формула в следующей ячейке не магическая. Частота задана как
`f(t) = 200 + 450 · t` (за 4 секунды это как раз даёт 2000 Гц), а в синус
подставляют не частоту, а **фазу** — интеграл частоты по времени:
`2π · (200 · t + 225 · t²)`. Отсюда и `(2000 - 200) / (2 · 4)` в коде: это
половина скорости роста, 225. Подставите свои начальную и конечную частоту —
получите свой свип.

In [ ]:
tc = np.arange(sr * 4) / sr                    # четыре секунды
# чирп: частота линейно растёт с 200 до 2000 Гц, поэтому в фазе появляется tc²
phase = 2 * np.pi * (200 * tc + (2000 - 200) / (2 * 4.0) * tc**2)
chirp = 0.5 * np.sin(phase)
print(chirp.shape, round(len(chirp) / sr, 2), "секунды")

In [ ]:
sc = np.abs(np.fft.rfft(chirp))       # спектр всей записи целиком, без деления на куски
fc = np.fft.rfftfreq(len(chirp), 1 / sr)
plt.plot(fc[:10000], sc[:10000])      # вместо пиков — широкая полка от 200 до 2000 Гц
plt.xlabel("частота, Гц")
plt.ylabel("амплитуда")
plt.show()

#### ❓ **Вопрос**: Спектр аккорда был двумя узкими пиками, а спектр чирпа — широкой полкой от 200 до 2000 Гц. Какой информации в этой полке не хватает?

<details>

<summary><strong>Ответ</strong></summary>

Не хватает **времени**. Полка честно говорит: «в записи встречались все частоты от 200 до 2000 Гц» — и это правда. Но точно такую же полку дала бы запись, где те же частоты звучали одновременно, или в обратном порядке, или вперемешку: преобразование Фурье усредняет по всей длительности.

А звук почти всегда меняется — речь, музыка, шум мотора. Отсюда идея следующего раздела: резать сигнал на короткие куски и считать спектр каждого отдельно.

</details>

## 8. STFT и спектрограмма

**STFT** (short-time Fourier transform) режет сигнал на перекрывающиеся **окна** и
считает спектр каждого. Параметров два:

- `n_fft` — длина окна в сэмплах: сколько сигнала видно за раз;
- `hop_length` — шаг между началами соседних окон.

Результат — комплексная матрица «частотные бины × кадры», она и называется
спектрограммой. Комплексное число здесь — это пара «амплитуда и фаза» данной
частоты в данном кадре; `np.abs` берёт амплитуду и выбрасывает фазу. Для
признаков так и делают: ухо и модели реагируют на то, какие частоты и насколько
громко звучали, а не на сдвиг волны. Обратно в звук по одной амплитуде уже не
вернуться точно — для этого фазу либо хранят, либо восстанавливают отдельными
алгоритмами.

Откуда берутся их значения. Окно длится `n_fft / sr` секунд, а шаг частотной
сетки равен `sr / n_fft`: одно улучшается только за счёт другого. Для речи обычно
берут окно в 20–40 мс (при 16 кГц это `n_fft = 512`), для музыки — длиннее. Наши
`n_fft = 2048` при `sr = 22050` дают окно 93 мс и шаг 10.8 Гц: для чирпа так
нагляднее (частотные полосы видно чётко), но для речи это окно слишком длинное —
не берите 2048 как «значение по умолчанию для всего». Степень двойки — потому что на ней быстрее всего
работает БПФ. `hop_length` делают в 2–4 раза меньше `n_fft` (у нас 512, то есть
`n_fft / 4`), чтобы окна перекрывались и картинка не «моргала» между кадрами:
чем меньше шаг, тем больше кадров и памяти. На оси времени это выглядит так:

```
сигнал  ─────────────────────────────────────────────►  t
окно 0  [───── n_fft = 2048 ─────]
окно 1       [───── 2048 ─────]        hop = 512
окно 2            [───── 2048 ─────]   (окна перекрываются на 3/4)
```

И ещё одна деталь, без которой формулировка «режет на окна» неполна: кусок перед
БПФ умножают на **оконную функцию** — у `librosa` это по умолчанию окно Ханна,
плавно спадающее к нулю по краям. Без него кусок обрывается посреди волны, и
такой разрыв БПФ размазывает по всему спектру («утечка спектра»): вместо узкого
пика получается пик с широкими юбками. Поэтому «окно» — это и кусок сигнала, и
функция, на которую его домножили; у `librosa` и `torchaudio` она по умолчанию
одна и та же (Ханна), а заменить её позволяют обе, только по-разному:
`librosa.stft(..., window="hamming")` принимает имя, а
`torchaudio.transforms.MelSpectrogram(..., window_fn=torch.hamming_window)` —
функцию.

Как STFT режет сигнал на окна

In [ ]:
# n_fft — длина окна в сэмплах, hop_length — на сколько окно сдвигается
S = librosa.stft(chirp, n_fft=2048, hop_length=512)
print(S.shape, S.dtype)                     # (частотные бины, кадры), числа комплексные
print(2048 // 2 + 1, 1 + len(chirp) // 512) # обе размерности считаются руками

#### ❓ **Вопрос**: Откуда в форме `(1025, 173)` взялись именно 1025 и 173?

<details>

<summary><strong>Ответ</strong></summary>

1025 — это `n_fft // 2 + 1`: спектр вещественного сигнала симметричен, поэтому хранится половина плюс нулевая частота. Ровно та же арифметика была у `rfft` в разделе 6.

173 — это `1 + len(chirp) // hop_length = 1 + 88200 // 512`: окна ставятся через каждые 512 сэмплов. По умолчанию `librosa` работает с `center=True` и дополняет сигнал по краям, поэтому кадр находится и для самого начала записи. Обе величины напечатаны второй строкой и совпали с формой.

Отсюда компромисс: увеличили `n_fft` — лучше разрешение по частоте, но хуже по времени (окно длиннее); уменьшили `hop_length` — больше кадров и больше памяти.

</details>

Амплитуды в спектрограмме различаются на порядки, и на линейной шкале виден
только самый громкий бин. Поэтому переходят к **децибелам** — логарифмической
шкале. `amplitude_to_db` берёт логарифм модуля, а `ref` задаёт точку отсчёта. С
`ref=np.max` ею становится максимум **этой** записи, так что 0 дБ — самое
громкое место в ней: удобно смотреть глазами, но абсолютная громкость при этом
теряется, и тихая запись выглядит как громкая. Когда признаки идут в модель и
записи надо сравнивать между собой, берут фиксированный `ref=1.0`.

In [ ]:
# ref=np.max — отсчитываем децибелы от самого громкого бина, поэтому максимум станет 0
S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
print(round(float(S_db.max()), 1), round(float(S_db.min()), 1))   # 0 и -80: пол задан top_db
print(S_db.shape, S_db.dtype)               # форма та же, числа теперь вещественные

Нижняя граница -80 дБ — это умолчание `top_db=80`. Децибелы амплитуды считаются
как `20 · log10(A / A_ref)`, поэтому -80 дБ — это `10^(-80/20) = 10^-4`: всё,
что тише максимума в 10 000 раз, обрезается, чтобы тишина не превращалась в
минус бесконечность.

Теперь `S_db` — обычный двумерный массив вещественных чисел, то есть картинка,
такая же, как в семинаре 15. Рисовать её умеет `librosa.display.specshow`: он
подписывает оси временем и частотой, а не номерами кадров и бинов.

In [ ]:
import librosa.display

# specshow сам подписывает оси, если сказать ему sr и hop_length
librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis="time", y_axis="hz")
plt.colorbar(format="%+2.0f dB")
plt.ylim(0, 3000)             # выше 3 кГц у чирпа ничего нет, незачем тратить картинку
plt.show()

Видна диагональ: частота растёт со временем. Проверим это численно — найдём
бин с максимумом энергии в первом и последнем кадре.

In [ ]:
bins = librosa.fft_frequencies(sr=sr, n_fft=2048)   # частота каждого бина в герцах
first = int(np.argmax(np.abs(S)[:, 0]))             # самый громкий бин первого кадра
last = int(np.argmax(np.abs(S)[:, -1]))             # и последнего
print(first, round(float(bins[first]), 1))          # около 200 Гц — начало чирпа
print(last, round(float(bins[last]), 1))            # около 2000 Гц — его конец

#### ❓ **Вопрос**: Номера бинов — 19 и 185. Как из них получаются 204.6 и 1991.8 Гц и почему не ровно 200 и 2000?

<details>

<summary><strong>Ответ</strong></summary>

Бины STFT равномерны по частоте с шагом `sr / n_fft = 22050 / 2048 ≈ 10.77` Гц, поэтому частота бина — это его номер, умноженный на шаг: `19 · 10.77 ≈ 204.6`, `185 · 10.77 ≈ 1991.8`. Именно такую таблицу и возвращает `librosa.fft_frequencies`.

Ровно 200 Гц на сетке нет — ближайший бин 19 и есть ответ, точнее одного бина STFT ответить не умеет (та же история, что с шагом 0.5 Гц в разделе 6, только окно грубее).

А вот 2000 Гц почти попали бы в бин 186 (2002.6 Гц), и всё же максимум встал на 185. Причина — не сетка, а окно: последний кадр центрирован на 88064-м сэмпле, то есть чуть раньше конца записи, и в его 2048 сэмплов попадает кусок чирпа с частотами вокруг 1990 Гц, а дальше — дополненная нулями пустота. Конца записи STFT «не видит» ровно так же, как не видит частоту точнее одного бина.

</details>

## 9. Мел-спектрограмма

1025 бинов — это много и неэффективно: человек различает 200 и 300 Гц легко, а 5000
и 5100 — почти никак. **Мел-шкала** — шкала частоты, растянутая внизу и сжатая
вверху, ближе к тому, как слышит ухо. Мел-спектрограмма собирает соседние бины STFT
в `n_mels` полос по этой шкале. Типовые значения — 40 (классическая речь),
64–80 (речь и звуковые события сегодня), 128 (музыка, где важны детали сверху):
больше полос — подробнее частота и больше чисел на вход модели, меньше — грубее
и дешевле.

<details><summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>Мел-шкала не выведена из физики — её измерили на людях. В 1937 году Стивенс,Волкман и Ньюман просили испытуемых подобрать тон «вдвое выше» заданного, иоказалось, что субъективное «вдвое выше» совпадает с удвоением частоты тольковнизу: разницу между 200 и 300 Гц слышно отчётливо, а 5000 и 5100 Гц для ухапочти одно и то же. Название — от слова melody.Отсюда практический вывод, ради которого мел-спектрограмма и живёт враспознавании речи: 1025 частотных бинов содержат уйму подробностей, которыечеловек всё равно не различает, а 64 мел-полосы оставляют ровно то, что несётсмысл. Модель учится быстрее просто потому, что ей не приходится самойдогадываться, какие различия важны.</details>

In [ ]:
# n_mels=64 — во столько полос сворачиваются 1025 бинов STFT
mel = librosa.feature.melspectrogram(y=chirp, sr=sr, n_fft=2048,
                                     hop_length=512, n_mels=64)
mel_db = librosa.power_to_db(mel, ref=np.max)   # power_to_db, а не amplitude: тут мощность
print(mel.shape, mel_db.shape)                  # кадров столько же, строк стало 64

Кадров столько же — 173, а частотных строк стало 64 вместо 1025. Весь конвейер
семинара в размерах (чирп, `sr = 22050`, `n_fft = 2048`, `hop_length = 512`):

```
сигнал      →  STFT          →  мел            →  дБ
(88200,)       (1025, 173)      (64, 173)         (64, 173)
сэмплы         бины × кадры     полосы × кадры    те же числа
```

Логарифм — всегда последний шаг: и в разделе 8, где мы взяли дБ прямо от STFT,
и здесь, где перед ним вклинилась мел-шкала. Собирать полосы уже из децибел
нельзя: складывать логарифмы вместо энергий — не то же самое.

Обратите внимание на `power_to_db`, а не `amplitude_to_db`: `melspectrogram`
возвращает **мощность** (квадрат амплитуды), и множитель перед логарифмом у неё
другой — `10 · log10(P / P_ref)` против `20 · log10(A / A_ref)` для амплитуд.
Множители подобраны так, что мощность и амплитуда одного сигнала дают одно и то
же число децибел (`P = A²`, а `log(A²) = 2 log A`). Перепутаете — шкала
растянется вдвое.

Посмотрим на саму шкалу. Центры полос — это внутренние точки сетки из `n_mels + 2`
мел-равномерных частот: крайние две задают начало первого и конец последнего
треугольного фильтра и центрами не являются.

In [ ]:
# центры мел-полос в герцах; крайние точки служебные, поэтому [1:-1]
mf = librosa.mel_frequencies(n_mels=64 + 2, fmax=sr / 2)[1:-1]
print(np.round(mf[:4], 1))                                        # внизу полосы частые
print(round(float(np.diff(mf)[0]), 1), round(float(np.diff(mf)[-1]), 1))  # шаг внизу и вверху

Словами это описать труднее, чем нарисовать. Каждая полоса — треугольный фильтр:
он берёт взвешенную сумму бинов STFT вокруг своего центра. Нарисуем восемь
таких фильтров (для 64 картинка была бы сплошной кашей).

In [ ]:
fb = librosa.filters.mel(sr=sr, n_fft=2048, n_mels=8)   # 8 полос вместо 64 — чтобы разглядеть
hz = librosa.fft_frequencies(sr=sr, n_fft=2048)
for row in fb:                # каждая строка матрицы — один треугольный фильтр
    plt.plot(hz, row)
plt.xlabel("частота, Гц")
plt.xlim(0, 8000)
plt.show()

Видно главное: слева треугольники узкие и стоят часто, справа — широкие и
редкие. Это и есть «растянуть низ, сжать верх»; сумма под каждым треугольником
и даёт одну строку мел-спектрограммы.

#### ❓ **Вопрос**: Шаг между соседними полосами внизу — 51.2 Гц, вверху — 537.8 Гц. Что это даёт признакам модели?

<details>

<summary><strong>Ответ</strong></summary>

Низ описан подробно, верх — грубо, одной полосой больше чем на 500 Гц. Для речи и музыки это ровно то, что нужно: основной тон голоса и первые **форманты** (усиленные голосовым трактом полосы частот, по которым мы и различаем гласные) лежат внизу, а наверху информации мало и она размазана.

Выигрыш виден по формам из предыдущей ячейки: вместо матрицы `1025 × 173` модель получает `64 × 173` — в 16 раз меньше чисел при почти той же полезной информации. Поэтому входом почти всех аудиомоделей служит именно мел-спектрограмма в дБ, а не сырой звук и не полный STFT.

</details>

In [ ]:
# y_axis="mel" — та же картинка, но ось частоты в мелах, а не в герцах
librosa.display.specshow(mel_db, sr=sr, hop_length=512,
                         x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.0f dB")
plt.show()

Та же диагональ, но ось частоты сжата сверху: на мел-шкале нижняя часть
занимает бо́льшую долю картинки. Это и есть «признаки, которые можно подать
модели», — и ровно это просят в домашке.

Сигналы у нас синтетические нарочно: их спектр известен заранее, и по картинке
видно, правильно ли всё посчитано. С речью или музыкой код ровно тот же —
меняется только файл, а диагональ чирпа превращается в полосы формант или
гармоники инструментов. Домашка как раз про настоящую запись.

## 10. То же самое в torchaudio

Если дальше идёт обучение на PyTorch, признаки удобнее считать сразу тензорами:
они попадают на то же устройство, что и модель, и участвуют в `Dataset` без
конвертаций. `torchaudio.load` возвращает **пару** «тензор, частота», как
`sf.read`, но тензор.

Оговорка про установку: начиная с версии 2.9 декодирование в `torchaudio`
вынесено в отдельный пакет `torchcodec`. Без него `load` падает с `ImportError`,
поэтому ставить надо оба: `uv add torchaudio torchcodec` (или
`pip install torchaudio torchcodec` — см. семинар 5).

In [ ]:
import torch
import torchaudio

wav, sr_t = torchaudio.load(work / "stereo.wav")   # тот же файл, что читали soundfile
print(type(wav).__name__, wav.shape, wav.dtype, sr_t)   # тензор float32, канал — первым

#### ❓ **Вопрос**: У `soundfile` тот же файл читался как `(44100, 2)` типа `float64`, а здесь — `torch.Size([2, 44100])` типа `float32`. Что из этого важно помнить?

<details>

<summary><strong>Ответ</strong></summary>

Оба отличия. Форма — `[каналы, сэмплы]`: канал первым, то же соглашение, что у `librosa` с `mono=False` в разделе 4, и противоположное `soundfile`. Тип — `float32`, а не `float64`: модели PyTorch по умолчанию работают в одинарной точности, и при переносе массива из numpy нужен явный `.float()`, иначе слой упадёт на несовпадении типов.

А вот ресемплинга здесь не произошло: `torchaudio.load` отдал 22050 — частоту как в файле, в отличие от `librosa.load`. Приводить частоту, если надо, придётся самим (`torchaudio.transforms.Resample`).

</details>

Соберём три соглашения в одну таблицу, чтобы не искать их по разделам:

| Загрузка | Моно | Стерео | Тип |
|----------|------|--------|-----|
| `sf.read` | `(N,)` | `(N, 2)` — канал последний | `float64` |
| `librosa.load(..., mono=False)` | `(N,)` | `(2, N)` — канал первый | `float32` |
| `torchaudio.load` | `[1, N]` | `[2, N]` — канал первый | `float32` |

Отдельно стоит запомнить `torchaudio`: у него даже моно — двумерный тензор
`[1, N]`, канал не исчезает.

Преобразования в `torchaudio` — это **модули** (`torch.nn.Module`), как слои
сети: объект создаётся один раз с параметрами, а потом вызывается на данных. Он
переносится на GPU вместе с моделью и работает сразу на батче.

Первый такой модуль — `Resample`: то самое приведение частоты, которое
`librosa.load` делал молча, а `torchaudio.load` не делает вовсе. Делают это
осознанно и по трём причинам: модель обучена на конкретной частоте (у речевых
это почти всегда 16 кГц) и другую не примет; в одном датасете записи приходят с
разными `sr`, а батч требует общей сетки; понижение частоты вдвое вдвое же
сокращает и память, и время счёта. Здесь для наглядности берём 8 кГц —
«телефонную» частоту.

In [ ]:
to_8k = torchaudio.transforms.Resample(orig_freq=sr_t, new_freq=8000)  # модуль создаём один раз
low = to_8k(wav)                                   # и дальше зовём как функцию
print(wav.shape, wav.shape[1] / sr_t)              # длительность до
print(low.shape, low.shape[1] / 8000)              # и после: сэмплов меньше, секунд столько же

Сэмплов стало меньше, длительность не изменилась — та же арифметика, что в
разделе 4, только теперь ресемплинг виден в коде явной строкой. Модулем задаётся
и мел-спектрограмма, причём сразу для нескольких записей.

In [ ]:
to_mel = torchaudio.transforms.MelSpectrogram(     # тот же расчёт, что у librosa выше
    sample_rate=sr, n_fft=2048, hop_length=512, n_mels=64)
batch = torch.from_numpy(np.stack([chirp, chirp * 0.5])).float()   # две записи одним батчем
print(batch.shape, to_mel(batch).shape)   # batch-измерение проходит насквозь

#### ❓ **Вопрос**: На вход слою пришёл тензор `[2, 88200]`, на выходе — `[2, 64, 173]`. Почему не понадобился цикл по записям?

<details>

<summary><strong>Ответ</strong></summary>

`MelSpectrogram` считает преобразование по **последнему** измерению, а все предыдущие трактует как батч. Поэтому `[2, 88200]` — это две записи по 88200 сэмплов, и к каждой приписалась матрица `64 × 173` с тем же смыслом, что дал `librosa` в разделе 9 (числа не совпадут: `librosa` по умолчанию нормирует мел-фильтры по Слейни и берёт слейни-шкалу мела, а `torchaudio` — без нормировки и по htk-шкале; `MelSpectrogram(..., norm="slaney", mel_scale="slaney")` сближает их).

Обратная сторона: тензор `[2, 88200]` неотличим от одной стереозаписи, которую `torchaudio.load` вернул строкой выше. Что означает первое измерение, знает только автор кода.

</details>

Осталась шкала дБ. В `torchaudio` это тоже модуль — `AmplitudeToDB`; ему нужно
сказать, мощность на входе или амплитуда (`stype`), потому что множитель перед
логарифмом разный. После `MelSpectrogram` на входе мощность.

In [ ]:
# stype="power" — на входе мощность (как у MelSpectrogram), а не амплитуда
to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)
feats = to_db(to_mel(batch))              # два модуля подряд — это и есть готовые признаки
print(feats.shape, round(float(feats.max()), 1), round(float(feats.min()), 1))

Это готовый вход модели: тот же `(64, 173)` на запись, что дал `librosa` в
разделе 9, только тензором и сразу на весь батч. Ровно эта пара модулей нужна
в задаче H4.

## 11. Видео: кадры во времени

Видео — это последовательность кадров плюс частота кадров **fps** (frames per
second). Один кадр — обычная картинка из семинара 15; новое здесь только время:

`число кадров = fps · длительность`

Это ровно та же конструкция, что спектрограмма: там запись резали на кадры по
времени и каждый кадр описывали столбиком частотных бинов, здесь кадры те же, а
столбик — пиксели. Поэтому и раскладка осей будет тем же самым вопросом «что
идёт первым измерением».

Соберём учебный клип: белый квадрат 8×8 едет вправо, 30 кадров при 10 кадрах в
секунду. На последних кадрах он частично уезжает за правый край — срез numpy
просто обрезается по ширине, и квадрат становится у́же.

In [ ]:
frames = np.zeros((30, 48, 64, 3), dtype=np.uint8)   # 30 чёрных кадров 64×48 RGB
for i in range(30):
    frames[i, 20:28, 2 * i:2 * i + 8] = 255          # белый квадрат едет вправо по 2 пикселя
print(frames.shape, frames.dtype)                    # (кадры, высота, ширина, каналы)

Форма — `(кадры, высота, ширина, каналы)`: к знакомой картинке `H × W × C`
приписано измерение времени. Кодировать в mp4 будем внешним `ffmpeg`: подаём сырые
байты на стандартный вход, получаем файл.

`ffmpeg` — не пакет Python, а системная программа: если её нет, ячейка упадёт с
`FileNotFoundError`. Ставится через пакетный менеджер (`apt install ffmpeg`,
`brew install ffmpeg`), в Colab уже стоит.

Флаги в команде ниже читаются так:

| Флаг | Что значит |
|------|------------|
| `-f rawvideo` | на входе не контейнер, а голые пиксели |
| `-pix_fmt rgb24` (до `-i`) | формат этих пикселей: 3 байта на точку |
| `-s 64x48`, `-r 10` | размер кадра и fps: в сыром потоке их взять неоткуда |
| `-i -` | читать со стандартного входа |
| `-c:v libx264` | кодек для записи, H.264 |
| `-pix_fmt yuv420p` (после `-c:v`) | формат уже внутри mp4, понятный плеерам |
| `-y`, `-loglevel error` | перезаписывать файл, молчать без ошибок |

При декодировании те же флаги работают в обратную сторону: `-i файл` на входе,
`-f rawvideo -pix_fmt rgb24 -` на выходе.

Аргументы всегда собираем **списком строк**, а не одной строкой: так `ffmpeg`
получит их ровно такими, какими мы написали, и путь с пробелом внутри не
развалится на два аргумента. Соблазнительное `"...".split()` этим и плохо. Проверить — `shutil.which("ffmpeg")`;
именно так делает `assets/make_data.py`, чтобы не падать на машине без него.

Контейнер и кодек

Сначала убедимся, что `ffmpeg` вообще есть: `shutil.which` возвращает путь к
программе или `None`. Лучше упасть здесь с понятным текстом, чем внутри
`subprocess` с `FileNotFoundError`.

In [ ]:
import shutil
import subprocess

# which вернёт путь к программе или None: упасть здесь понятнее, чем внутри subprocess
assert shutil.which("ffmpeg"), "нет ffmpeg: поставьте его (apt install ffmpeg)"

In [ ]:
clip = work / "clip.mp4"
# -f rawvideo: на вход идут голые пиксели, поэтому размер кадра и fps сообщаем руками
cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24",
       "-s", "64x48", "-r", "10", "-i", "-", "-c:v", "libx264",   # -i - : читать из stdin
       "-pix_fmt", "yuv420p", str(clip)]                          # -c:v : каким кодеком сжать
subprocess.run(cmd, input=frames.tobytes(), check=True)
print(clip.stat().st_size, "байт")

Метаданные читает `ffprobe` — он не декодирует видео, а смотрит заголовки,
поэтому работает мгновенно даже на большом файле. Это аналог `sf.info` для звука.

In [ ]:
# ffprobe читает только заголовок, поэтому отвечает мгновенно на файле любого размера
probe = ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
         "stream=width,height,r_frame_rate,nb_frames", "-of", "csv=p=0"]
out = subprocess.run(probe + [str(clip)],
                     capture_output=True, text=True, check=True)
print(out.stdout.strip())        # ширина, высота, кадров в секунду, число кадров

Обратный путь — декодировать в массив кадров. Самый переносимый способ тот же
`ffmpeg`: попросим его выдать сырые пиксели на стандартный выход и разберём
буфер.

In [ ]:
decode = ["ffmpeg", "-v", "error", "-i", str(clip),      # обратный путь: mp4 → пиксели
          "-f", "rawvideo", "-pix_fmt", "rgb24", "-"]   # "-" в конце: писать в stdout
raw = subprocess.run(decode, capture_output=True, check=True).stdout
back = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 48, 64, 3)  # байты → массив без копии
print(back.shape, back.dtype)
print(int(np.abs(back.astype(int) - frames.astype(int)).max()))   # 0 = кодек ничего не потерял

#### ❓ **Вопрос**: Форма `back` — `(30, 48, 64, 3)`, а `ffprobe` напечатал `64,48,10/1,30`. Как получить длительность клипа и почему `reshape(-1, 48, 64, 3)` вообще сработал?

<details>

<summary><strong>Ответ</strong></summary>

Длительность — `30 / 10 = 3.0` секунды: число кадров, делённое на fps (`nb_frames = 30`, `r_frame_rate = 10/1`). Ровно та же формула, что для звука в разделе 1, только вместо сэмплов кадры.

`reshape` сработал потому, что размер кадра мы знали заранее: `ffmpeg` отдаёт сплошной поток байт без всяких разделителей, и разложить его на кадры можно, только зная высоту, ширину и формат пикселя (`rgb24` — 3 байта на пиксель). Возьмёте не те числа — `reshape` либо упадёт, либо тихо соберёт кашу.

</details>

Кадры теперь обычный массив — посмотрим на один глазами, это та же картинка
из семинара 15.

In [ ]:
plt.imshow(back[10])          # декодированный кадр — обычная картинка H×W×C
plt.title("кадр 10")
plt.show()

Расхождение с исходными кадрами нулевое: наш синтетический клип из плоских
чёрных и белых прямоугольников кодек H.264 передал точно. В общем случае так не
бывает — H.264 сжимает **с потерями** и вдобавок хранит цветность в `yuv420p`
прорежённой вдвое по каждой оси — отсчётов цветности вчетверо меньше, чем
пикселей, так что реальная съёмка после кодирования по пикселям
не совпадёт. Нужны кадры бит-в-бит — храните их без потерь (`-c:v ffv1` или
последовательность png).

Про `torchvision`: в свежих версиях (0.28) декодирование видео из него **убрано** —
`torchvision.io` остался только про изображения (`decode_image`, `read_image`), а
тензор кадров теперь отдаёт тот же `torchcodec`, что нужен и `torchaudio`. Зато
трансформы `torchvision` никуда не делись, и к декодированным кадрам они
применимы — это мы и сделаем следующим шагом.

In [ ]:
from torchcodec.decoders import VideoDecoder

decoder = VideoDecoder(str(clip))            # не читает файл целиком, кадры берутся по запросу
print(len(decoder), decoder.metadata.average_fps)
print(decoder[:].shape, decoder[10].shape)   # срез — весь клип, индекс — один кадр

#### ❓ **Вопрос**: `ffmpeg` дал форму `(30, 48, 64, 3)`, а `VideoDecoder` — `[30, 3, 48, 64]`. Что изменилось и почему так сделано?

<details>

<summary><strong>Ответ</strong></summary>

Переставлены измерения: `(T, H, W, C)` против `[T, C, H, W]` — канал уехал с последнего места на второе. Это стандартная раскладка входа свёрточных слоёв PyTorch (`NCHW`), поэтому тензор из `VideoDecoder` можно подавать в модель без `permute`.

Заодно обратите внимание: у звука `torchaudio` тоже кладёт канал перед временем (`[каналы, сэмплы]`, раздел 10), а `ffmpeg` и `soundfile` — после. Одно и то же правило: перед использованием печатаем `.shape`.

</details>

Раз кадры — это батч картинок в раскладке `NCHW`, к ним применимо всё из
семинара 15 — там же стоит и сам `torchvision` (`uv add torchvision`).
Трансформы принимают такой тензор целиком: уменьшим все 30 кадров до 32×24
одним вызовом.

In [ ]:
from torchvision.transforms import v2

small = v2.Resize((24, 32))(decoder[:])   # преобразование само проходит по батчу кадров
print(small.shape, small.dtype)           # уменьшились только высота и ширина

#### ❓ **Вопрос**: Почему `Resize` не пришлось звать в цикле по кадрам и что стало бы с этим кодом, будь тензор в раскладке `(T, H, W, C)`?

<details>

<summary><strong>Ответ</strong></summary>

Трансформы `torchvision` работают по **двум последним** измерениям (`H` и `W`), а всё, что левее, считают батчем — ровно как `MelSpectrogram` в разделе 10 считал батчем всё левее последнего измерения. Поэтому `[30, 3, 48, 64]` — это тридцать картинок, и цикл не нужен.

В раскладке `(T, H, W, C)` последними двумя измерениями оказались бы ширина и каналы: `Resize` молча растянул бы не то, что нужно. Такой массив (например, из `ffmpeg`) сначала переводят в тензор и делают `permute(0, 3, 1, 2)`.

</details>

## Итог

- Звук в памяти — массив чисел от -1 до 1; смысл ему даёт частота дискретизации:
  `длительность = число сэмплов / sr`.
- `soundfile`: `sf.read` → `(массив, sr)`, форма стерео `(сэмплы, каналы)`,
  `sf.info` читает метаданные без декодирования.
- `librosa.load` по умолчанию ресемплирует к 22050 Гц и сводит в моно — пишите
  `sr=None`, если этого не хотите; форма стерео у него `(каналы, сэмплы)`.
- Громкость правят делением на пик, а не умножением: за пределами -1..1 начинается
  клиппинг, а вместе с ним лишние гармоники.
- Спектр всей записи описывает только неизменный сигнал. Для меняющегося нужен
  **STFT**: форма `(n_fft // 2 + 1, 1 + N // hop_length)`, шаг по частоте
  `sr / n_fft`.
- Шкала дБ (`amplitude_to_db` для амплитуд, `power_to_db` для мощности) делает
  спектрограмму читаемой; мел-шкала сжимает 1025 бинов до `n_mels` полос — это и
  есть типичный вход аудиомодели.
- `torchaudio`: `load` → `(тензор [каналы, сэмплы] float32, sr)`, `transforms` —
  модули (`Resample`, `MelSpectrogram`), работающие на батче и на GPU. Нужен
  отдельный пакет `torchcodec`: `pip install torchaudio` его не подтягивает.
- Видео — кадры плюс fps; метаданные быстрее всего смотреть через `ffprobe`,
  кадры разбирать через `ffmpeg` в `(T, H, W, C)` или `VideoDecoder` в
  `[T, C, H, W]`. Кадры в `[T, C, H, W]` — обычный батч картинок, и трансформы
  `torchvision` работают на них без изменений.

Домашка — спектрограмма аудиозаписи. Задачи семинара — в `tasks.md`, учебные
данные готовит `python3 assets/make_data.py`: бинарных файлов в репозитории нет,
всё генерируется детерминированно, поэтому числа в тестовых примерах у всех
одинаковые.